# 🧠 Cognita AI — GPU Inference & RAG Server

This notebook runs the **FastAPI inference & RAG backend** for Cognita AI on a GPU instance (Google Colab / Kaggle).

### Features:
- **Qwen3-4B / Qwen3.5-9B** causal language model on GPU.
- **RAG Foundation**: Dedicated dense embedding model + PostgreSQL/pgvector retrieval.
- **Query Router**: Automatic dispatch across `COMPANY_RAG`, `WEB_SEARCH` (Tavily), `BOTH`, and `DIRECT_LLM`.
- **SSE Streaming**: Real-time token streaming with `status`, `thinking`, and `sources` metadata.
- **Public Tunneling**: Exposes API via Ngrok for your local or deployed Node/React frontend.

---

## 📦 Cell 1: Install Dependencies

In [ ]:
# Install all required inference, RAG, and tunnel dependencies
!pip install -q accelerate bitsandbytes fastapi uvicorn pyngrok python-dotenv tavily-python transformers sentence-transformers pypdf python-docx psycopg2-binary pgvector python-multipart

print("✅ Dependencies installed successfully.")

## 🔑 Cell 2: Configure Environment & Secrets
Set your secrets via Kaggle Secrets / Colab User Data or directly below.

In [ ]:
import os

# Read from Kaggle Secrets if available, otherwise set environment variables
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["NGROK_AUTHTOKEN"] = user_secrets.get_secret("NGROK_AUTHTOKEN")
    try:
        os.environ["TAVILY_API_KEY"] = user_secrets.get_secret("TAVILY_API_KEY")
    except Exception:
        pass
    try:
        os.environ["DATABASE_URL"] = user_secrets.get_secret("DATABASE_URL")
    except Exception:
        pass
    print("✅ Loaded secrets from Kaggle Secrets.")
except Exception:
    # Fallback: fill in directly if not using Kaggle Secrets
    os.environ.setdefault("NGROK_AUTHTOKEN", "your-ngrok-token-here")
    os.environ.setdefault("TAVILY_API_KEY", "")
    os.environ.setdefault("DATABASE_URL", "")
    print("ℹ️ Using environment variables / defaults.")

os.environ.setdefault("MODELS_TO_LOAD", "qwen3-4b")
os.environ.setdefault("DEFAULT_MODEL", "qwen3-4b")
os.environ.setdefault("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
os.environ.setdefault("EMBEDDING_DIMENSION", "384")
os.environ.setdefault("CHUNK_SIZE", "500")
os.environ.setdefault("CHUNK_OVERLAP", "100")
os.environ.setdefault("RAG_TOP_K", "4")
os.environ.setdefault("RAG_MIN_SCORE", "0.5")
os.environ.setdefault("PORT", "8000")

## 🤖 Cell 3: Load Qwen & Embedding Model

In [ ]:
from load_models import load_configured_models, load_embedding_model

# 1. Load LLM on GPU
print("Loading Qwen model...")
load_configured_models()

# 2. Load Embedding model for RAG
print("Loading Embedding model...")
load_embedding_model()

print("✅ Models loaded and ready.")

## 🚀 Cell 4: Start FastAPI Server in Background

In [ ]:
import subprocess
import time
import requests

print("Starting FastAPI server...")
server_process = subprocess.Popen(
    ["python", "run_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for server to become healthy
time.sleep(3)
for _ in range(10):
    try:
        res = requests.get("http://127.0.0.1:8000/health")
        if res.status_code == 200:
            print("✅ FastAPI server is healthy:", res.json())
            break
    except Exception:
        time.sleep(1)
else:
    print("⚠️ Server is still starting or check output log.")

## 🌐 Cell 5: Expose via Ngrok Tunnel
Copy the printed public URL to `INFERENCE_HOST` or `OLLAMA_HOST` in your Node backend's `.env`.

In [ ]:
from pyngrok import ngrok

authtoken = os.environ.get("NGROK_AUTHTOKEN")
if not authtoken:
    raise ValueError("Please set NGROK_AUTHTOKEN in Cell 2 or Kaggle Secrets.")

ngrok.set_auth_token(authtoken)

# Disconnect any existing tunnels
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

port = int(os.environ.get("PORT", "8000"))
public_tunnel = ngrok.connect(addr=port, proto="http")

print("=" * 60)
print(f"🚀 Cognita API Public URL: {public_tunnel.public_url}")
print("=" * 60)
print("📋 Copy this URL into your backend's .env file:")
print(f"INFERENCE_HOST={public_tunnel.public_url}")
print("=" * 60)